## Задача 1.

In [1]:
import numpy as np
from decimal import Decimal, getcontext
import math
from functools import lru_cache

getcontext().prec = 100  

@lru_cache(None)
def factorial_dec(n: int) -> Decimal:
    if n < 0:
        raise ValueError("Negative factorial not defined")
    return Decimal(1) if n in [0, 1] else Decimal(n) * factorial_dec(n - 1)

@lru_cache(None)
def comb_dec(n: int, k: int) -> Decimal:
    return Decimal(0) if k < 0 or k > n else factorial_dec(n) / (factorial_dec(k) * factorial_dec(n - k))

@lru_cache(None)
def D(n: int, j: int) -> Decimal:
    if j == 0:
        return Decimal(1)
    _ = D(n, j - 1)
    return _compute_all_D_up_to(n, j)[j]

@lru_cache(None)
def _compute_all_D_up_to(n: int, J: int):
    results = [Decimal(0)] * (J + 1)
    results[0] = Decimal(1)
    for j in range(J):
        s = Decimal(0)
        for k in range(n + 2 * j + 1):
            for l in range(j + 1):
                s += (
                    Decimal((-1) ** k)
                    * results[l]
                    * comb_dec(n + 2 * l, k - (j - l))
                    * Decimal((n + 2 * j - 2 * k) ** (n + 2 * j + 2))
                    / (Decimal(2) ** (n + 2 * j + 2) * factorial_dec(n + 2 * j + 2))
                )
        results[j + 1] = s
    return results

@lru_cache(None)
def A(k: int, n: int, m: int) -> Decimal:
    s = Decimal(0)
    for l in range(m + 1):
        s += D(n, l) * comb_dec(n + 2 * l, k - m + l) * Decimal((-1) ** (k - m))
    return s

@lru_cache(None)
def W_list(m: int):
    result = []
    for k in range(2 * m + 1):
        s = sum(A(k, 2 * n_, m - n_) / (Decimal(2) ** (2 * n_) * factorial_dec(2 * n_ + 1)) for n_ in range(m + 1))
        result.append(s)
    return result

def build_function_values_diff_scheme(f, a: float, b: float, J: int, m: int):
    h = (b - a) / J
    return [f(a + (j + 0.5) * h) for j in range(-m, J - 1 + m + 1)], h

def integrate_by_diff_scheme(f, a: float, b: float, J: int, m: int):
    y_values, h = build_function_values_diff_scheme(f, a, b, J, m)
    W = [float(wd) for wd in W_list(m)]
    total_sum = sum(sum(W[m - k] * y_values[j + k + m] for k in range(-m, m + 1)) for j in range(J))
    return h * total_sum, len(y_values)

def build_function_values_simpson(f, a: float, b: float, n: int):
    h = (b - a) / n
    return [f(a + i * h) for i in range(n + 1)], h

def simpson_by_calls(f, a: float, b: float, calls: int):
    n = max(2, calls - 1 - (calls - 1) % 2)
    y, h = build_function_values_simpson(f, a, b, n)
    return (h / 3) * (y[0] + y[n] + 4 * sum(y[i] for i in range(1, n, 2)) + 2 * sum(y[i] for i in range(2, n, 2))), n + 1

def main():
    def f1(x: float) -> float:
        return 0.5 if x == 0 else math.sin(x / 2) / (np.exp(x) - 1)

    def get_almost_exact_for_f1():
        return simpson_by_calls(f1, -1.0, 1.0, 10_000_001)[0]

    exact_val_1 = get_almost_exact_for_f1()
    a1, b1 = -1.0, 1.0

    print("\nПример")
    print(f"Интеграл: I1 = ∫[-1,1] sin(x/2)/(e^x-1) dx ≈ {exact_val_1:.8f}\n")

    for m in [3, 4, 5, 6, 7]:
        print(f"--- m={m} ---")
        for J in [2, 4, 8, 16]:
            diff_approx, diff_calls = integrate_by_diff_scheme(f1, a1, b1, J, m)
            simpson_approx, simp_real_calls = simpson_by_calls(f1, a1, b1, diff_calls)
            print(f" J={J}, calls={diff_calls:2d} | "
                  f"DiffScheme={diff_approx:.8f}, err={abs(diff_approx - exact_val_1):.2e} | "
                  f"Simpson={simpson_approx:.8f}, err={abs(simpson_approx - exact_val_1):.2e}  "
                  f"(n={simp_real_calls - 1} subintrvls)")
        print()

if __name__ == "__main__":
    main()



Пример
Интеграл: I1 = ∫[-1,1] sin(x/2)/(e^x-1) dx ≈ 1.01303924

--- m=3 ---
 J=2, calls= 8 | DiffScheme=1.01303490, err=4.34e-06 | Simpson=1.01303303, err=6.20e-06  (n=6 subintrvls)
 J=4, calls=10 | DiffScheme=1.01303922, err=1.96e-08 | Simpson=1.01303728, err=1.95e-06  (n=8 subintrvls)
 J=8, calls=14 | DiffScheme=1.01303924, err=7.95e-11 | Simpson=1.01303885, err=3.84e-07  (n=12 subintrvls)
 J=16, calls=22 | DiffScheme=1.01303924, err=3.06e-13 | Simpson=1.01303919, err=4.97e-08  (n=20 subintrvls)

--- m=4 ---
 J=2, calls=10 | DiffScheme=1.01303816, err=1.07e-06 | Simpson=1.01303728, err=1.95e-06  (n=8 subintrvls)
 J=4, calls=12 | DiffScheme=1.01303923, err=1.45e-09 | Simpson=1.01303844, err=7.98e-07  (n=10 subintrvls)
 J=8, calls=16 | DiffScheme=1.01303924, err=1.53e-12 | Simpson=1.01303903, err=2.07e-07  (n=14 subintrvls)
 J=16, calls=24 | DiffScheme=1.01303924, err=6.22e-15 | Simpson=1.01303920, err=3.40e-08  (n=22 subintrvls)

--- m=5 ---
 J=2, calls=12 | DiffScheme=1.01303887, er

## Задача 2.



In [2]:
@lru_cache(None)
def W_list(m: int):
    """
    Коэффициенты W_k^m (k=0..2m) в Decimal.

      W_k^m = sum_{n=0}^m [ A_{k,2n}^{m-n} / (2^(2n)*(2n+1)!) ].
    """
    result = []
    for k in range(2*m + 1):
        s = Decimal(0)
        for n_ in range(m+1):
            a_val = A(k, 2*n_, m - n_)
            denom = (Decimal(2)**(2*n_) * factorial_dec(2*n_+1))
            s += a_val / denom
        result.append(s)
    return result

@lru_cache(None)
def I_list(m: int):
    result = []
    for l in range(2*m+1):
        s = Decimal(0)
        for k in range(l+1):
            s += W_list(m)[2*m-k]
        result.append(s)
    print(result)
    return result

def f(x):
    if x == 1.0:
        return 1
    return 1/x -np.log(x) + np.log(x-1)

def integratePartGamma(j_m):
    # Первообразная f
    return -((j_m-0.5)*np.log(1-1/(j_m+0.5))+1)

def GammaSearch(f_, m, j_m):
    s = 0
    #Integral part
    s += integratePartGamma(j_m)

    print("Integral part = ", s)

    # Partial Summ
    for j in range(1, j_m-m+1):
        s += f_(j)
    print("Partial Sum + Integral Part = ", s)

    # I part
    

    I_dec = I_list(m)
    I = [float(id) for id in I_dec]

    for j in range(j_m-m+1, j_m+m+1):
        
        s += (1-I[j+m-j_m-1])*f_(j)
    
    return(s)
    



In [3]:
Gamma = Decimal(0.5772156649015328606065120900824024310421593359399235988057672348848677267776646709369470632917467495146314472498070824809605040144865428362241739976449235362535003337429373377376739427925952582470949160087352039481656708532331517766115286211995015079847937450857057400299213547861)
j_mList = np.array([10e1, 10e3, 10e5, 10e7])
mList = np.array([5, 7, 9, 11])

for m in mList:
    for j_m in j_mList:

        print(f"m = {m}, j_m = {j_m} :", np.abs(GammaSearch(f, 7, 1000)-float(Gamma)))


Integral part =  -0.0004999167083316047
Partial Sum + Integral Part =  0.5772191883535537
[Decimal('3.920487188820469138256698882801175570135005761108053877013312639414932183891619517721810490769926370E-7'), Decimal('-0.000007086554852531810495488226418561515563279231709566806568570237000572097573861242291577388579152247548'), Decimal('0.00006303618834393500912262279656459550639444819338999233179127359021538915718809898704078598258492438366'), Decimal('-0.0003766549564375071550846772683192436278856031942451695538115291201710954797374550460970214056633809712'), Decimal('0.001788177072404827529224066991372987845650984980790977263640402970208966681629820959626956099619238946'), Decimal('-0.008156106427634993329408709493034889860286685683511080336477161873987270812667638064463461288858114255'), Decimal('0.05933621505754702731783223268011627799987588347376707165066953426741786530146318506106865895225683590'), Decimal('0.94066378494245297268216776731988372200012411652623292834933046573258213

# Задача 3.

Вычислить сумму ряда

 
 
с
с точностью 
, применяя метод Куммера

In [4]:
import numpy as np

def f(x):
    return (x*x + 1) * np.cos(2*x) / (x**4 + x*x + 1)

def g(x):
    return np.cos(2*x) / x**2

gSumm = np.pi**2/6 - np.pi + 1  

def CummerRule(f_, g_, i=1, m=10):
    s = sum(f_(j) - g_(j) for j in range(i, m+1))
    return s + gSumm

for j in [10, 100, 1000]:
    sum_kummer = CummerRule(f, g, 1, j)
    print(f"Сумма правилом Куммера, k = {j:>4}: {sum_kummer:.13f}")


Сумма правилом Куммера, k =   10: -0.3512657639505
Сумма правилом Куммера, k =  100: -0.3512653858792
Сумма правилом Куммера, k = 1000: -0.3512653858792


# Задача 4.

Взять предел

 
 
где

  
Поиграться с выбором шага.

In [5]:
import numpy as np
from decimal import Decimal, getcontext
from math import factorial

getcontext().prec = 100  

# Вспомогательные функции для факториала и биномиальных коэффициентов
def factorial_dec(n: int) -> Decimal:
    return Decimal(1) if n in [0, 1] else Decimal(n) * factorial_dec(n - 1)

def comb_dec(n: int, k: int) -> Decimal:
    return Decimal(0) if k < 0 or k > n else factorial_dec(n) / (factorial_dec(k) * factorial_dec(n - k))

def A(k: int, n: int, m: int) -> Decimal:
    s = sum(D(n, l) * comb_dec(n + 2 * l, k - m + l) * Decimal((-1) ** (k - m)) for l in range(m + 1))
    return s

def D(n: int, j: int) -> Decimal:
    return Decimal(1) if j == 0 else _compute_all_D_up_to(n, j)[j]

def _compute_all_D_up_to(n: int, J: int):
    results = [Decimal(0)] * (J + 1)
    results[0] = Decimal(1)
    for j in range(J):
        results[j + 1] = sum(
            (-1) ** k * results[l] * comb_dec(n + 2 * l, k - (j - l)) * (n + 2 * j - 2 * k) ** (n + 2 * j + 2) /
            (2 ** (n + 2 * j + 2) * factorial_dec(n + 2 * j + 2))
            for k in range(n + 2 * j + 1) for l in range(j + 1)
        )
    return results

def y_of_x(x: float) -> float:
    return np.cosh(np.sqrt(abs(x))) if x > 0 else np.cos(np.sqrt(abs(x))) if x < 0 else 2.0

def f_of_t(t: float) -> float:
    return 0 if t == 0 else (y_of_x(t) - 1.0) / t

def r_of_n(n: int) -> int:
    return n % 2

def limit_by_sym_scheme(m: int, h: float, f) -> float:
    N = 2 * m + 1
    j_min, j_max = -N, N
    offset = -j_min

    y_array = [f((2 * j - 1) * h / 2.0) for j in range(j_min, j_max + 1)]

    S = 0.0
    for n_ in range(N + 1):
        rr = r_of_n(n_)
        alpha_int = int(m - (n_ - rr) / 2.0)

        factor_n = (-1) ** n_ / (4 ** n_ * factorial(n_))
        max_k = 2 * m + rr

        for k_ in range(max_k + 1):
            j_idx = (2 * m + rr) - 2 * k_
            S += float(A(k_, n_, alpha_int)) * factor_n * y_array[j_idx + offset]

    return S

# Запуск вычисления предела с разными значениями h
def test():
    m_values = [1 + 2 * j for j in range(5)]
    h_values = [10 ** (-p) for p in range(4, 11)]

    for m in m_values:
        print(f"\n--- m = {m} ---")
        for h in h_values:
            val = limit_by_sym_scheme(m, h, f_of_t)
            error = abs(val - 0.5)
            print(f"h = {h:.0e}, ошибка = {error:.3e}")

test()



--- m = 1 ---
h = 1e-04, ошибка = 4.167e-06
h = 1e-05, ошибка = 4.167e-07
h = 1e-06, ошибка = 4.166e-08
h = 1e-07, ошибка = 4.828e-09
h = 1e-08, ошибка = 3.039e-09
h = 1e-09, ошибка = 1.999e-08
h = 1e-10, ошибка = 4.137e-08

--- m = 3 ---
h = 1e-04, ошибка = 4.167e-06
h = 1e-05, ошибка = 4.167e-07
h = 1e-06, ошибка = 4.166e-08
h = 1e-07, ошибка = 4.806e-09
h = 1e-08, ошибка = 3.073e-09
h = 1e-09, ошибка = 1.909e-08
h = 1e-10, ошибка = 4.137e-08

--- m = 5 ---
h = 1e-04, ошибка = 4.167e-06
h = 1e-05, ошибка = 4.167e-07
h = 1e-06, ошибка = 4.166e-08
h = 1e-07, ошибка = 4.800e-09
h = 1e-08, ошибка = 3.091e-09
h = 1e-09, ошибка = 1.889e-08
h = 1e-10, ошибка = 4.137e-08

--- m = 7 ---
h = 1e-04, ошибка = 4.167e-06
h = 1e-05, ошибка = 4.167e-07
h = 1e-06, ошибка = 4.166e-08
h = 1e-07, ошибка = 4.797e-09
h = 1e-08, ошибка = 3.102e-09
h = 1e-09, ошибка = 1.882e-08
h = 1e-10, ошибка = 4.137e-08

--- m = 9 ---
h = 1e-04, ошибка = 4.167e-06
h = 1e-05, ошибка = 4.167e-07
h = 1e-06, ошибка = 4.166